# CurveAudit road-state training

Open this notebook from **Mac A** at [colab.research.google.com](https://colab.research.google.com). Upload this file. There is no API key.

Runtime → Change runtime type → **H100** if the menu lets you select it. On Colab Pro, H100 is often greyed out and limited to Pro+. Choose **A100** and start. Do not spend the night retrying H100. This network is MobileNetV3-Small. An A100 finishes it.

The GitHub repo is private, so Colab cannot clone it. On Mac A:

```bash
cd ~/Code/CurveAudit
zip -r /tmp/curveaudit-ml.zip ml/labels.py ml/roadnet.py ml/train_surface.py
```

Upload `/tmp/curveaudit-ml.zip` with the Colab file browser, into `/content`.

RSCD (Mendeley, CC BY 4.0, DOI `10.17632/w86hvkrzc5`) is about 10 GB. Download it in the browser on Mac A. Do not download the million-image RSXD release. Put the class folders in Google Drive at `My Drive/CurveAudit/rscd`, or upload them to `/content/rscd`. The folder `--data` points at must contain the 27 class folders, not a zip.

Keep this tab open. When the last cell finishes, download `roadsurface.pt` and `roadsurface.metrics.json` to Mac A at `ml/runs/` before the runtime is deleted. Then, on Mac A:

```bash
python3 ml/convert_coreml.py ml/runs/roadsurface.pt ios/CurveAudit/RoadSurface.mlpackage
```


In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime, Change runtime type, H100 if selectable, otherwise A100.')
name = torch.cuda.get_device_name(0)
print(name)
if 'H100' not in name and 'A100' not in name:
    print('This is not an H100 or A100. If the menu offers one of those, switch to it and rerun. Otherwise keep going.')


In [ ]:
from pathlib import Path
import zipfile
zip_path = Path('/content/curveaudit-ml.zip')
if not zip_path.exists():
    raise SystemExit('Upload /tmp/curveaudit-ml.zip from Mac A into /content.')
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall('/content')
needed = ['ml/labels.py', 'ml/roadnet.py', 'ml/train_surface.py']
missing = [path for path in needed if not Path('/content', path).exists()]
if missing:
    raise SystemExit('Zip is missing: ' + ', '.join(missing))
print('training code is in /content/ml')


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as error:
    print('Drive was not mounted.', error)

candidates = [
    Path('/content/rscd'),
    Path('/content/drive/MyDrive/CurveAudit/rscd'),
    Path('/content/drive/MyDrive/CurveAudit/rscd/train-set'),
]
source = next((path for path in candidates if path.is_dir()), None)
if source is None:
    raise SystemExit('Put the 27 RSCD class folders in My Drive/CurveAudit/rscd or /content/rscd.')
if (source / 'train-set').is_dir():
    source = source / 'train-set'
print('class folders:', sorted(path.name for path in source.iterdir() if path.is_dir())[:30])
local = Path('/content/rscd-local')
if not local.exists():
    print('Copying images onto the Colab disk so training does not read Drive one file at a time.')
    import shutil
    shutil.copytree(source, local)
print('training images:', local)


In [ ]:
import subprocess, sys
command = [
    sys.executable, '/content/ml/train_surface.py',
    '--data', '/content/rscd-local',
    '--epochs', '5',
    '--batch', '128',
    '--out', '/content/roadsurface.pt',
]
subprocess.check_call(command)


In [ ]:
from pathlib import Path
import shutil
drive_dir = Path('/content/drive/MyDrive/CurveAudit')
if Path('/content/drive').exists():
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy('/content/roadsurface.pt', drive_dir / 'roadsurface.pt')
    metrics = Path('/content/roadsurface.metrics.json')
    if metrics.exists():
        shutil.copy(metrics, drive_dir / metrics.name)
    print('Copied the checkpoint to', drive_dir)
from google.colab import files
files.download('/content/roadsurface.pt')
metrics = '/content/roadsurface.metrics.json'
if Path(metrics).exists():
    files.download(metrics)
